# Author Performance Feature Engineering

Creates the author-level dataset used by the Author Performance dashboard. Duplicate exploratory cells were consolidated, and unused Pareto calculations were excluded.

## 1. Load and validate the author-level dataset

Each row represents one unique author.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed")
INPUT_FILE = DATA_DIR / "author_features.csv"
OUTPUT_FILE = DATA_DIR / "author_features_powerbi.csv"

author_df = pd.read_csv(INPUT_FILE)
print(f"Rows: {len(author_df):,}")
author_df.head()

In [ ]:
required_columns = [
    "author_id", "total_views", "like_count", "finish_count",
    "avg_duration_time", "unique_videos", "unique_musics",
    "unique_real_time", "activities"
]

missing_columns = [col for col in required_columns if col not in author_df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")
if author_df["author_id"].duplicated().any():
    raise ValueError("author_id is not unique; expected one row per author.")

print("Unique authors:", author_df["author_id"].nunique())
print("Total views:", author_df["total_views"].sum())
print("Total likes:", author_df["like_count"].sum())

## 2. Create author performance metrics

Rates are stored as decimals between 0 and 1. Format them as percentages in Power BI.

In [ ]:
author_df["finish_rate"] = np.where(
    author_df["total_views"] > 0,
    author_df["finish_count"] / author_df["total_views"],
    0
)

author_df["like_rate"] = np.where(
    author_df["total_views"] > 0,
    author_df["like_count"] / author_df["total_views"],
    0
)

author_df["views_per_video"] = np.where(
    author_df["unique_videos"] > 0,
    author_df["total_views"] / author_df["unique_videos"],
    0
)

author_df["has_views"] = (author_df["total_views"] > 0).astype("int8")
author_df["has_likes"] = (author_df["like_count"] > 0).astype("int8")

## 3. Create Power BI distribution bins

Numeric order columns keep category labels in the intended business order.

In [ ]:
view_bins = [0, 1, 5, 10, 20, 50, float("inf")]
view_labels = [
    "1 view", "2–5 views", "6–10 views",
    "11–20 views", "21–50 views", "51+ views"
]
author_df["view_bin"] = pd.cut(
    author_df["total_views"], bins=view_bins, labels=view_labels,
    right=True, include_lowest=True
)
view_bin_order = {label: order for order, label in enumerate(view_labels, start=1)}
author_df["view_bin_order"] = author_df["view_bin"].astype("string").map(view_bin_order)

like_bins = [-1, 0, 1, float("inf")]
like_labels = ["0 likes", "1 like", "2+ likes"]
author_df["like_bin"] = pd.cut(
    author_df["like_count"], bins=like_bins, labels=like_labels, right=True
)
like_bin_order = {label: order for order, label in enumerate(like_labels, start=1)}
author_df["like_bin_order"] = author_df["like_bin"].astype("string").map(like_bin_order)

In [ ]:
view_distribution = (
    author_df["view_bin"].value_counts(sort=False).rename("authors").to_frame()
)
view_distribution["percentage"] = (view_distribution["authors"] / len(author_df) * 100).round(2)

like_distribution = (
    author_df["like_bin"].value_counts(sort=False).rename("authors").to_frame()
)
like_distribution["percentage"] = (like_distribution["authors"] / len(author_df) * 100).round(2)

print("View distribution")
display(view_distribution)
print("Like distribution")
display(like_distribution)

## 4. Validate and export the Power BI table

In [ ]:
powerbi_columns = [
    "author_id", "total_views", "like_count", "finish_count",
    "finish_rate", "like_rate", "avg_duration_time",
    "unique_videos", "views_per_video", "unique_musics",
    "unique_real_time", "activities", "has_views", "has_likes",
    "view_bin", "view_bin_order", "like_bin", "like_bin_order"
]

author_features_powerbi = author_df[powerbi_columns].copy()

checks = {
    "rows": len(author_features_powerbi),
    "unique_authors": author_features_powerbi["author_id"].nunique(),
    "duplicate_author_ids": int(author_features_powerbi["author_id"].duplicated().sum()),
    "missing_view_bins": int(author_features_powerbi["view_bin"].isna().sum()),
    "missing_like_bins": int(author_features_powerbi["like_bin"].isna().sum()),
    "invalid_finish_rates": int((~author_features_powerbi["finish_rate"].between(0, 1)).sum()),
    "invalid_like_rates": int((~author_features_powerbi["like_rate"].between(0, 1)).sum())
}
print(checks)

if any(checks[key] != 0 for key in [
    "duplicate_author_ids", "missing_view_bins", "missing_like_bins",
    "invalid_finish_rates", "invalid_like_rates"
]):
    raise ValueError("Validation failed; review the checks above.")

author_features_powerbi.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Exported: {OUTPUT_FILE}")
author_features_powerbi.head()